## <font color='red'> INSTRUCTIONS </font>

<b> 
1. Write your code only in cells below the "WRITE CODE BELOW" title. Do not modify the code below the "DO NOT MODIFY" title. <br>
2. The expected data types of the output answers for each question are given in the last cell through assertion statements. Your answers must match these expected output data types. Hint: Many of the answers need to be a Python dictionary. Consider methods like to_dict() to convert a Pandas Series to a dictionary. <br>
3. The answers are then written to a JSON file named my_results_PA1.json. You can compare this with the provided expected output file "expected_results_PA1.json". <br>
4. After you complete writing your code, click "Kernel -> Restart Kernel and Run All Cells" on the top toolbar. There should NOT be any syntax/runtime errors, otherwise points will be deducted. <br>
5. For submitting your solution, first download your notebook by clicking "File -> Download". Rename the file as &ltTEAM_ID&gt.ipynb" and upload to Canvas.</b>


## <font color='red'> DO NOT MODIFY </font>

In [1]:
import time
import json
import dask
import dask.dataframe as dd
import pandas as pd
import ast
import re
from dask.distributed import Client
import ctypes
import numpy as np

def trim_memory() -> int:
    """
    helps to fix any memory leaks.
    """
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client = Client("127.0.0.1:8786")
client.run(trim_memory)
client = client.restart()
print(client)

None


In [ ]:
start = time.time()

## <font color='blue'> WRITE CODE BELOW </font>

In [2]:
user_reviews = dd.read_csv('user_reviews.csv')
products = dd.read_csv('products.csv', dtype={'asin': 'object'})

num_reviews = len(user_reviews)
num_products = len(products)

In [ ]:
## Question 1
# Time: 30 seconds
# start = time.time()
null_percent = ((user_reviews.isna().sum() / num_reviews) * 100).round(2).compute()
# num_nulls = user_reviews.isna().sum().compute()
# num_nulls = np.round((num_nulls / user_reviews.shape[0]) * 100, 2)
ans1 = null_percent.to_dict()
# end = time.time()
ans1

In [ ]:
## Question 2
# Time: 18.55 seconds
# start = time.time()
null_percent = ((products.isna().sum() / num_products) * 100).round(2).compute()
ans2 = null_percent.to_dict()
# end = time.time()
ans2

In [16]:
# ## Question 3
# Time: 56.6 seconds
start = time.time()
# Needed to speed this up, trying to index by asin
user_reviews_indexed = user_reviews.set_index('asin')
products_indexed = products.set_index('asin')
product_price = products_indexed[['price']]

# Need to join so that all user reviews have the price of the associated product
# Might need to remove this later
joined = user_reviews_indexed.join(product_price, how='left')

# This is a 2x2 dataframe
corr = joined[['price', 'overall']].corr(method='pearson').compute()
ans3 = corr['price'].iloc[1].round(2)
end = time.time()
print(end - start)
ans3

94.1762318611145


np.float64(-0.01)

In [ ]:
## Question 4
# Time: 11.4 seconds
# start = time.time()
desc = products['price'].describe().compute()
ans4 = {'mean': desc['mean'], 'std': desc['std'], 'min': desc['min'], 'max': desc['max'],'median': desc['50%']}
# end = time.time()
# print(end - start)
ans4

In [ ]:
## Question 5
# Time: 13.67 seconds

# Idea: Process the category column from the products table and get first value from each list
# Groupby this processed column and use .count()
# Sort in non-increasing order
   
# start = time.time()
first_cat = products['categories'].dropna().str.extract(r"\[\['([^']+)")[0]
products = products.assign(first_cat=first_cat)
selected = products[['asin', 'first_cat']].groupby(by='first_cat').count().sort_values(by='asin',ascending=False).compute()
ans5 = selected.to_dict()
# end = time.time()
# print(end-start)
selected.head()

In [17]:
## Question 6
# time is 1 min
# Remove this later, defined for question 3
# start = time.time()
joined = user_reviews_indexed.merge(product_price,how='left',indicator=True)
ans6 = 1 if len(joined[joined['_merge']=='left_only']) > 0 else 0
# end = time.time()
# print(end - start)
ans6

57.56613349914551


1

In [41]:
## Question 7
import ast

def process(related_str):
    related_dict = ast.literal_eval(related_str)
    all_asins = []
    for v in related_dict.values():
        if isinstance(v, list):
            all_asins.extend(v)
    return all_asins

start = time.time()
is_dangling = 0
full_asins = products['asin'].dropna().values
for r in range(num_products):
    current = products.loc[r]
    curr_asins = process(current['related'])
    exploded=curr_asins.explode()
    exploded.index.name="or_index"
    m=exlpoded.isin(full_asins)
    check=dd.concat([exploded.reset_index(), m.rename('in_full_asins')], axis=1)
    a=check.groupby('or_index')['in_full_asins'].all()
    a=a.compute()
    is_dangling=a
        #if not curr_asins.issubset(full_asins):
     #   is_dangling = 1
      #  break
        
ans7 = is_dangling
end = time.time()
print(end - start)
ans7

/home/ubuntu/dask_env/lib/python3.10/site-packages/dask/dataframe/dask_expr/_collection.py:1430: UserWarning: Dask currently has limited support for converting pandas extension dtypes to arrays. Converting string to object dtype.
  warnings.warn(


ValueError: malformed node or string: Dask Series Structure:
npartitions=152
    string
       ...
     ...  
       ...
       ...
Dask Name: getitem, 4 expressions
Expr=(LocUnknown(frame=ArrowStringConversion(frame=FromMapProjectable(9db3c2b)), iindexer=slice(0, 0, None)))['related']

In [42]:
import ast
import time

def process(related_str):
    try:
        related_dict = ast.literal_eval(related_str)
        all_asins = []
        for v in related_dict.values():
            if isinstance(v, list):
                all_asins.extend(v)
        return all_asins
    except (ValueError, SyntaxError):
        return []

start = time.time()

is_dangling = 0
full_asins = products['asin'].dropna().values

# Iterate over the Dask DataFrame in chunks (adjustable with `compute` if needed)
for r in range(num_products):
    current = products.loc[r]
    
    # Ensure 'related' is a string and apply the process function
    related_asins = process(current['related'])
    
    # Use pandas for explosion of lists
    exploded = pd.Series(related_asins).explode()
    exploded.index.name = "or_index"
    
    # Convert exploded list back into a Dask DataFrame
    m = exploded.isin(full_asins)
    check = dd.from_pandas(pd.concat([exploded.reset_index(), m.rename('in_full_asins')], axis=1), npartitions=4)
    
    # Perform the groupby operation
    a = check.groupby('or_index')['in_full_asins'].all()
    a = a.compute()  # Compute the result
    is_dangling = a

ans7 = is_dangling

end = time.time()
print(end - start)
ans7


/home/ubuntu/dask_env/lib/python3.10/site-packages/dask/dataframe/dask_expr/_collection.py:1430: UserWarning: Dask currently has limited support for converting pandas extension dtypes to arrays. Converting string to object dtype.
  warnings.warn(


ValueError: Cannot call len() on object with unknown chunk size.

A possible solution: https://docs.dask.org/en/latest/array-chunks.html#unknown-chunks
Summary: to compute chunks sizes, use

   x.compute_chunk_sizes()  # for Dask Array `x`
   ddf.to_dask_array(lengths=True)  # for Dask DataFrame `ddf`

In [48]:
import itertools
import ast

current = products.loc[1]['related'].compute()
current = current.astype(str)
my_dict = ast.literal_eval(current)
flattened_list = list(itertools.chain(*my_dict.values()))
flattened_list

ValueError: malformed node or string: 1    {'also_viewed': ['B0036FO6SI', 'B000KL8ODE', '...
1    {'also_bought': ['0078049997', '0072893249', '...
1    {'also_bought': ['0803278314', '0415909937', '...
1    {'also_bought': ['030726677X', '0415253985', '...
1                                                 <NA>
                           ...                        
1    {'also_viewed': ['B00JU708M4', 'B0053O1PY8', '...
1    {'also_bought': ['B00ITF9RV6', 'B00IF7VE5U', '...
1    {'also_bought': ['B00ICM8YAQ', 'B00IP3WTTY', '...
1    {'also_bought': ['B00KVRAUAW', 'B00LCDM2C8', '...
1    {'also_viewed': ['B003WQLN8M', 'B00HQDYNDO', '...
Name: related, Length: 152, dtype: object

In [46]:
type(current)

pandas.core.series.Series

In [ ]:
### read in the 'user_reviews.csv' and 'products.csv' files, perform your calculations and place the answers in variables ans1 - ans7.


# substitute 'None' with the outputs from your calculations. 
# The expected output types can be seen in the assertion statements below
ans1 = ans1
ans2 = ans2
ans3 = ans3
ans4 = ans4
ans5 = ans5
ans6 = an6
ans7 = None

## <font color='red'> DO NOT MODIFY </font>

In [ ]:
end = time.time()

In [ ]:
print(f"execution time = {end-start}s")

In [ ]:
# DO NOT MODIFY
assert type(ans1) == dict, f"answer to question 1 must be a dictionary like {{'reviewerID':0.2, ..}}, got type = {type(ans1)}"
assert type(ans2) == dict, f"answer to question 2 must be a dictionary like {{'asin':0.2, ..}}, got type = {type(ans2)}"
assert type(ans3) == float, f"answer to question 3 must be a float like 0.8, got type = {type(ans3)}"
assert type(ans4) == dict, f"answer to question 4 must be a dictionary like {{'mean':0.4,'max':0.6,'median':0.6...}}, got type = {type(ans4)}"
assert type(ans5) == dict, f"answer to question 5 must be a dictionary, got type = {type(ans5)}"         
assert ans6 == 0 or ans6==1, f"answer to question 6 must be 0 or 1, got value = {ans6}" 
assert ans7 == 0 or ans7==1, f"answer to question 7 must be 0 or 1, got value = {ans7}" 

ans_dict = {
    "q1": ans1,
    "q2": ans2,
    "q3": ans3,
    "q4": ans4,
    "q5": ans5,
    "q6": ans6,
    "q7": ans7,
    "runtime": end-start
}
with open('my_results_PA1.json', 'w') as outfile: json.dump(ans_dict, outfile)         